# Matrix Operations

In [5]:
import logging

from systemds.context import SystemDSContext

logger = logging.getLogger('simple_example')
logger.setLevel(logging.INFO)

# Create a context and if necessary (no SystemDS py4j instance running)
# it starts a subprocess which does the execution in SystemDS
with SystemDSContext() as sds:
    # Full generates a matrix completely filled with one number.
    # Generate a 5x10 matrix filled with 4.2
    m = sds.full((5, 10), 4.20)
    # multiply with scalar. Nothing is executed yet!
    m_res = m * 3.1
    # Do the calculation in SystemDS by calling compute().
    # The returned value is an numpy array that can be directly printed.
    logger.info(m_res.compute())
    # context will automatically be closed and process stopped

26/05/06 14:56:07 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
INFO:simple_example:[[13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02]
 [13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02]
 [13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02]
 [13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02]
 [13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02 13.02]]


In [9]:
import logging

import numpy as np
from systemds.context import SystemDSContext

logger = logging.getLogger('simple_example')
logger.setLevel(logging.INFO)

# create a random array
m1 = np.array(np.random.randint(100, size=5 * 5) + 1.01, dtype=np.double)
m1 = m1.reshape(5, 5)
# create another random array
m2 = np.array(np.random.randint(5, size=5 * 5) + 1, dtype=np.double)
m2 = m2.reshape(5, 5)

# Create a context
with SystemDSContext() as sds:
    # element-wise matrix multiplication, note that nothing is executed yet!
    m_res = sds.from_numpy(m1) * sds.from_numpy(m2)
    # lets do the actual computation in SystemDS! The result is an numpy array
    m_res_np = m_res.compute()
    logger.info(m_res_np)

26/05/06 15:02:21 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 15:02:21 WARNING SystemDSContext: Deprecated method from_numpy. Use from_py instead.
26/05/06 15:02:21 WARNING SystemDSContext: Deprecated method from_numpy. Use from_py instead.
INFO:simple_example:[[ 50.01 125.05 160.05  32.01 204.04]
 [ 37.01  81.03 268.04 110.02   6.03]
 [ 39.01 244.04 430.05 140.04 368.04]
 [246.03 154.02 267.03 104.04  72.02]
 [ 14.02 186.02 475.05 168.02 315.05]]


# Algorithms as built-in functions

In [10]:
import logging

import numpy as np
from systemds.context import SystemDSContext
from systemds.operator.algorithm import l2svm

logger = logging.getLogger('simple_example')
logger.setLevel(logging.INFO)

# Set a seed
np.random.seed(0)
# Generate random features and labels in numpy
# This can easily be exchanged with a data set.
features = np.array(np.random.randint(
    100, size=10 * 10) + 1.01, dtype=np.double)
features = features.reshape(10, 10)
labels = np.zeros((10, 1))

# l2svm labels can only be 0 or 1
for i in range(10):
    if np.random.random() > 0.5:
        labels[i][0] = 1

# compute our model
with SystemDSContext() as sds:
    model = l2svm(sds.from_numpy(features),
                  sds.from_numpy(labels), verbose=False).compute()
    logger.info(model)

26/05/06 15:05:20 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 15:05:20 WARNING SystemDSContext: Deprecated method from_numpy. Use from_py instead.
26/05/06 15:05:20 WARNING SystemDSContext: Deprecated method from_numpy. Use from_py instead.
INFO:simple_example:[[ 0.02033445]
 [-0.00324092]
 [ 0.0014692 ]
 [ 0.02649209]
 [-0.00616902]
 [-0.0095087 ]
 [ 0.01039221]
 [-0.0011352 ]
 [-0.01686351]
 [-0.03839821]]


In [11]:
import logging

from systemds.context import SystemDSContext
from systemds.operator.algorithm import l2svm

logger = logging.getLogger('simple_example')
logger.setLevel(logging.INFO)

with SystemDSContext() as sds:
    # Generate 10 by 10 matrix with values in range 0 to 100.
    features2 = sds.rand(10, 10, 0, 100)
    # Add value to all cells in features
    features2 += 1.1
    # Generate labels of all ones and zeros
    labels2 = sds.rand(10, 1, 1, 1, sparsity=0.5)

    model2 = l2svm(features2, labels2, verbose=False).compute()
    logger.info(model2)

26/05/06 15:07:21 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
INFO:simple_example:[[-0.02674835]
 [-0.01684308]
 [ 0.02429331]
 [ 0.03332674]
 [-0.00411165]
 [ 0.00514915]
 [ 0.01035419]
 [ 0.00611115]
 [-0.02550657]
 [-0.0117999 ]]
